> **Revision note (AI-service integration).** This is the original notebook, unchanged in logic, with two additions at the very end:
> (1) a cell that adds the **donation-oriented category** (`donation_category`) to `food_category_mapping.json` so the FastAPI matching/usable-time modules can use it, and (2) a cell that packages the artifacts as `food_classifier_artifacts.zip` for copying into `ai_service/saved_models/food_classifier/`.
> **Status: this notebook has NOT been executed in the delivery environment** (it needs a Colab GPU and the 4.6 GB Food-101 download). The same train/export logic was verified end-to-end on synthetic data with `scripts/train_food_classifier.py --smoke-synthetic` (plumbing only, metrics meaningless). No accuracy figure is claimed.

# Food Image Recognition with MobileNetV2 Transfer Learning (Food-101)

**Research project:** *Dynamic AI-Assisted Surplus Food Redistribution Under Time and Demand Uncertainty*

This notebook builds, trains, evaluates, saves and tests a **food-type image classifier** that helps a donor fill in a food-donation listing.

### What the model does
Given one food photo it returns: a predicted food name, a broader food category, the top-3 classes with confidence scores, and a **"needs manual verification"** flag when it is unsure.

### What the model does NOT do
It does **not** estimate food quantity, freshness, remaining usable time, or food safety. A photo classifier cannot know those things. Those must come from the donor / other parts of your system. It is also **not** your project's dynamic matching algorithm, and a Food-101 classifier by itself is **not** a research contribution.

### How to run
1. `Runtime > Change runtime type > T4 GPU`.
2. Run the cells **top to bottom**. The first run downloads Food-101 (~4.6 GB), which takes a while.
3. **Recommended first run:** set `SMOKE_TEST = True` in the config cell. It runs a tiny version of everything in a few minutes to verify the pipeline. Metrics from a smoke test are **meaningless**. Then set it back to `False` and re-run for the real experiment.

### Two small deviations from your outline (and why)
* The **food-category mapping** (your item 15) comes *before* the **prediction function** (item 14), because `predict_food()` must return the category.
* The **software tests** (item 12) run *after* saving/reloading, so they test the exact saved artifacts you will deploy.

### A few design decisions worth knowing
* **Preprocessing lives in one file** (`food_inference.py`), imported by training, evaluation, prediction and your future API, so train and inference can't drift apart.
* The MobileNetV2 rescaling (0..255 → −1..1) is a **layer inside the model**, so any code that feeds a 224×224 RGB image gets correct preprocessing automatically.
* Augmentation layers are **inside the model** too, but they are only active during training.
* The official Food-101 **test** set is touched exactly **once**, for the final evaluation. Model selection and the uncertainty threshold use the **validation** set only.

## Step 1 · Environment and GPU verification

**What it does:** prints library versions and checks that a GPU is visible.
**Why:** training MobileNetV2 on ~68k images on a CPU is impractically slow. Verifying versions first makes later errors easier to diagnose.

**Common errors**
* `GPUs found: []` → `Runtime > Change runtime type > Hardware accelerator: T4 GPU`, then re-run.
* `ModuleNotFoundError: tensorflow_datasets` → the cell installs it for you (it does **not** touch your TensorFlow version).

In [ ]:
import sys, subprocess
print("Python:", sys.version.split()[0])

import tensorflow as tf
print("TensorFlow:", tf.__version__)

try:
    import tensorflow_datasets as tfds
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-datasets"])
    import tensorflow_datasets as tfds
print("TensorFlow Datasets:", tfds.__version__)

from tensorflow import keras
print("Keras:", keras.__version__)
if int(keras.__version__.split(".")[0]) < 3:
    print("NOTE: this notebook was written for Keras 3 (current Colab). It may still work on Keras 2.")

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)
if gpus:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print("GPU:", out.stdout.strip())
else:
    print("WARNING: no GPU. Training will be extremely slow. Enable a GPU runtime before continuing.")

## Step 2 · Imports, configuration and reproducibility

**What it does:** imports libraries, defines **all settings in one dictionary** (`CFG`), creates output folders and fixes random seeds.

**Why seeds:** the same seed gives the same train/validation split and (mostly) the same augmentation randomness. GPU training is still not bit-for-bit deterministic, so tiny run-to-run differences are normal.

**Important parameters**
| Setting | Meaning |
|---|---|
| `SEED` | Random seed used everywhere. |
| `BATCH_SIZE` | Images per training step. Lower it (e.g. 32) if you get GPU out-of-memory errors. |
| `VAL_FRACTION` | Share of the *official training set* held out for validation (0.10 = 10%). |
| `HEAD_LR` / `FINE_TUNE_LR` | Learning rates. The fine-tuning rate is much smaller to avoid destroying pretrained features. |
| `FINE_TUNE_AT` | Backbone layers with index `< FINE_TUNE_AT` stay frozen. MobileNetV2 has ~154 layers. |
| `SMOKE_TEST` | `True` = tiny fast run to check that the code works (results meaningless). |
| `TRAIN_CACHE_FILE` | Optional disk cache of resized training images (`None` = off). Needs ~10 GB disk and makes epochs 2+ faster. |

**Common errors:** none expected here. If you change settings after running later cells, re-run from this cell downward.

In [ ]:
import os, json, random, shutil, time, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, f1_score, accuracy_score,
                             classification_report, confusion_matrix)
from tensorflow.keras import layers
from IPython.display import display

# ------------------------------- configuration -------------------------------
CFG = {
    "SEED": 42,
    "IMG_SIZE": 224,
    "BATCH_SIZE": 64,
    "VAL_FRACTION": 0.10,
    "DROPOUT": 0.3,
    "HEAD_LR": 1e-3,
    "FINE_TUNE_LR": 1e-5,
    "FINE_TUNE_AT": 100,
    "PIPELINE_CHECK_EPOCHS": 2,     # short first run to verify the pipeline
    "EXTRA_HEAD_EPOCHS": 6,         # optional continuation if validation loss is still improving
    "FINE_TUNE_EPOCHS": 10,
    "EARLY_STOP_PATIENCE": 3,
    "SHUFFLE_BUFFER": 1000,
    "TRAIN_CACHE_FILE": None,       # e.g. "/content/train_cache" to cache resized images on disk
    "TARGET_ACCEPTED_ACC": 0.90,    # used to pick the uncertainty threshold on the VALIDATION set
    "MARGIN_THRESHOLD": 0.10,       # flag if top-1 minus top-2 probability is below this (heuristic)
    "SAVE_TO_DRIVE": True,          # optional Google Drive export (Step 17)
    "DRIVE_FOLDER": "/content/drive/MyDrive/food_recognition_artifacts",
    "SMOKE_TEST": False,            # <-- set True for a fast pipeline check
    "SMOKE_BATCHES": {"train": 20, "val": 5, "test": 10},
}
if CFG["SMOKE_TEST"]:
    CFG.update(PIPELINE_CHECK_EPOCHS=1, EXTRA_HEAD_EPOCHS=0, FINE_TUNE_EPOCHS=1)
    print("*** SMOKE TEST MODE: all metrics below are NOT meaningful ***")

SEED = CFG["SEED"]
IMG_SIZE = CFG["IMG_SIZE"]
BATCH_SIZE = CFG["BATCH_SIZE"]

# --------------------------------- folders -----------------------------------
ARTIFACT_DIR = "/content/food_model_artifacts"
CKPT_DIR = os.path.join(ARTIFACT_DIR, "checkpoints")
PLOT_DIR = os.path.join(ARTIFACT_DIR, "plots")
SAMPLE_DIR = "/content/sample_test_images"
for d in (ARTIFACT_DIR, CKPT_DIR, PLOT_DIR, SAMPLE_DIR):
    os.makedirs(d, exist_ok=True)

# ------------------------------- reproducibility -----------------------------
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)   # seeds Python, NumPy and TensorFlow together

# --------------------------------- helpers -----------------------------------
def _json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    return str(o)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)

def maybe_limit(ds, split):
    """In smoke-test mode only use a few batches so the whole notebook runs quickly."""
    return ds.take(CFG["SMOKE_BATCHES"][split]) if CFG["SMOKE_TEST"] else ds

print("Configuration loaded. Artifacts folder:", ARTIFACT_DIR)

## Step 3 · Dataset: Food-101 (download and load)

**About the dataset**
* **Food-101** has 101 food categories with 1,000 images each (101,000 images). Each class has 750 training and 250 test images.
* **Official source:** ETH Zurich Computer Vision Lab: <https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/>
* **Citation:** Bossard, Guillaumin & Van Gool, *"Food-101 – Mining Discriminative Components with Random Forests"*, ECCV 2014.
* **TensorFlow Datasets catalog entry:** <https://www.tensorflow.org/datasets/catalog/food101>
* The authors state that the **training images were deliberately left uncleaned** (some noise: very saturated colours and some wrong labels), while the **test images were manually cleaned**.

⚠️ **Naming trap:** in TFDS the splits are called `train` (75,750 images) and `validation` (25,250 images). **TFDS's `validation` split is the official Food-101 *test* set.** The code below loads it as `raw_test` and never uses it for training or tuning. Our own validation set is carved out of the official *train* split in Step 5.

**What the code does**
* `tfds.load(...)` downloads (~4.6 GB, one-off) and prepares the data.
* `shuffle_files=False` keeps the order fixed, which our split logic relies on.
* `SkipDecoding()` keeps images as raw JPEG bytes. We decode them later inside the fast `tf.data` pipeline and can read labels without decoding any image.

**Common errors**
* *No space left on device* → Colab disk is full; restart the runtime to clear it.
* *Download/connection error* → re-run the cell; TFDS resumes.
* The split names or image counts differ → the cell prints a warning. Check the TFDS catalog page above.

In [ ]:
t0 = time.time()
(raw_train_full, raw_test), ds_info = tfds.load(
    "food101",
    split=["train", "validation"],      # NOTE: TFDS "validation" == official Food-101 TEST split
    shuffle_files=False,                # fixed order => reproducible split
    with_info=True,
    decoders={"image": tfds.decode.SkipDecoding()},   # keep JPEG bytes; decode later in tf.data
)
print(f"Loaded in {time.time() - t0:.0f}s")

CLASS_NAMES = ds_info.features["label"].names
NUM_CLASSES = len(CLASS_NAMES)
n_train_official = ds_info.splits["train"].num_examples
n_test_official = ds_info.splits["validation"].num_examples
print("Classes:", NUM_CLASSES)
print("Official train images:", n_train_official, "| official test images:", n_test_official)

assert NUM_CLASSES == 101, f"Expected 101 classes, got {NUM_CLASSES}"
if (n_train_official, n_test_official) != (75750, 25250):
    warnings.warn("Split sizes differ from the documented 75,750 / 25,250. Check the TFDS catalog.")

print("\nThe 101 classes:")
print(", ".join(CLASS_NAMES))

## Step 4 · Dataset inspection and visualization

**What it does:** reads every label (cheap, because images are not decoded), counts images per class, and shows sample images with their labels.

**Why:** always look at your data before training. Check that classes are balanced, that labels match images, and what the images actually look like (single plated dishes, close-ups, restaurant photos).

**Class balance and limitations (read this, it matters for your write-up)**
* Official splits are **balanced**: exactly 750 train and 250 test images per class, so class imbalance is *not* a problem here (we still use a *stratified* split so validation keeps this balance).
* The **training labels are noisy** by design, so validation accuracy (which comes from the same noisy pool) may be slightly pessimistic.
* Food-101 is mostly **single, plated, restaurant-style dishes** photographed one at a time. Real donation photos (bulk trays, containers, packaged food, mixed dishes, poor lighting) will look different, so **expect lower accuracy in the real world than on the Food-101 test set**.
* The **101 classes are limited** and skewed towards Western dishes (plus some East Asian ones). Many common dishes (for example a typical Indian thali, dal, roti, biryani) have **no class** at all, and the model will confidently pick the *nearest* wrong class. This is why manual correction and an "other / unknown" path are essential in your app.

In [ ]:
def extract_labels(raw_ds):
    """Read only the integer labels (no image decoding) -> 1-D int64 array in dataset order."""
    ds = raw_ds.map(lambda ex: ex["label"], num_parallel_calls=tf.data.AUTOTUNE).batch(4096)
    return np.concatenate([b.numpy() for b in ds]).astype(np.int64)

t0 = time.time()
train_full_labels = extract_labels(raw_train_full)
test_labels_official = extract_labels(raw_test)
print(f"Labels read in {time.time() - t0:.0f}s")
assert len(train_full_labels) == n_train_official and len(test_labels_official) == n_test_official

counts_train = np.bincount(train_full_labels, minlength=NUM_CLASSES)
counts_test = np.bincount(test_labels_official, minlength=NUM_CLASSES)
print("Images per class (official train): min", counts_train.min(), "max", counts_train.max())
print("Images per class (official test):  min", counts_test.min(), "max", counts_test.max())

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(NUM_CLASSES), counts_train, label="official train")
ax.bar(range(NUM_CLASSES), counts_test, label="official test")
ax.set_xlabel("class index"); ax.set_ylabel("images"); ax.set_title("Images per class (balanced)")
ax.legend(); plt.tight_layout(); plt.show()

# sample images with labels (decode just 12 examples)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, ex in zip(axes.ravel(), tfds.as_numpy(raw_train_full.take(12))):
    img = tf.io.decode_image(ex["image"], channels=3, expand_animations=False).numpy()
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{CLASS_NAMES[ex['label']]}\n{img.shape[1]}x{img.shape[0]}", fontsize=9)
plt.suptitle("Sample Food-101 training images (original size)"); plt.tight_layout(); plt.show()

## Step 5 · Train / validation / test preparation (no data leakage)

**The three sets**
| Set | Source | Used for |
|---|---|---|
| **Train** | 90% of the official train split | learning weights |
| **Validation** | 10% of the official train split | early stopping, picking the best checkpoint, choosing the uncertainty threshold |
| **Test** | official test split (TFDS `validation`) | **one final evaluation only** |

**How leakage is avoided**
* We split by **image index** with `train_test_split(..., stratify=labels, random_state=SEED)`: every image lands in exactly one of train/val, and every class keeps the same proportion in both. The code asserts there is **no overlap**.
* The test set is a separate official split and is never used until Step 12.
* Honest caveat: we can only guarantee no *index* overlap. Food-101 does not publish a near-duplicate audit, so near-identical photos *could* exist across splits.

**What the code does:** builds boolean masks, then filters the raw dataset with them (`enumerate` + `filter`). Because the order is fixed (`shuffle_files=False`), index *i* always refers to the same image. The cell **verifies** this alignment by comparing labels read through the filter against the expected labels.

**Common errors:** *"alignment check failed"* → you re-ran Step 3 with different settings; re-run Steps 3–5 in order.

In [ ]:
all_idx = np.arange(len(train_full_labels))
train_idx, val_idx = train_test_split(
    all_idx, test_size=CFG["VAL_FRACTION"], stratify=train_full_labels,
    random_state=SEED, shuffle=True)

# --- leakage checks ---
assert len(set(train_idx) & set(val_idx)) == 0, "Train/val overlap!"
assert len(train_idx) + len(val_idx) == len(all_idx)

train_mask = np.zeros(len(all_idx), dtype=bool); train_mask[train_idx] = True
val_mask = ~train_mask

def subset_by_mask(raw_ds, mask):
    """Keep the examples whose position in the (fixed-order) dataset has mask==True."""
    mask_t = tf.constant(mask)
    return (raw_ds.enumerate()
                  .filter(lambda i, ex: tf.gather(mask_t, i))
                  .map(lambda i, ex: ex))

raw_train = subset_by_mask(raw_train_full, train_mask)
raw_val = subset_by_mask(raw_train_full, val_mask)

# --- verify that the filtered dataset really contains the intended images ---
val_labels_via_pipeline = extract_labels(raw_val)
assert np.array_equal(val_labels_via_pipeline, train_full_labels[val_mask]), "alignment check failed"
print("Alignment check passed.")

sizes = pd.DataFrame({
    "split": ["train", "validation", "test (official, untouched until Step 12)"],
    "images": [int(train_mask.sum()), int(val_mask.sum()), len(test_labels_official)],
    "min_per_class": [np.bincount(train_full_labels[train_mask], minlength=NUM_CLASSES).min(),
                      np.bincount(train_full_labels[val_mask], minlength=NUM_CLASSES).min(),
                      counts_test.min()],
    "max_per_class": [np.bincount(train_full_labels[train_mask], minlength=NUM_CLASSES).max(),
                      np.bincount(train_full_labels[val_mask], minlength=NUM_CLASSES).max(),
                      counts_test.max()],
})
display(sizes)

## Step 6 · Image preprocessing and efficient `tf.data` pipelines

**Every preprocessing step**
1. **Decode** the JPEG bytes into pixels, forcing **3 channels (RGB)**. This handles grayscale or odd formats.
2. **Resize to 224×224** (`tf.image.resize`, bilinear, with antialiasing so down-scaling doesn't create jagged artefacts). MobileNetV2 was pretrained at this size. We stretch rather than crop so the whole dish stays visible.
3. **Round and store as `uint8`.** This keeps data small and makes the result identical every time.
4. **Rescale pixels to [−1, 1]**, the exact convention of MobileNetV2 (`x / 127.5 − 1`). This is the first layer **inside the model** (Step 8), so it is automatically identical in training and inference.

Steps 1–3 are implemented once in `to_model_input()` inside `food_inference.py`. **Training, evaluation and `predict_food()` all use that same function.**

**Pipeline performance tricks**
* `num_parallel_calls=AUTOTUNE`: decode images on several CPU cores.
* `.batch()`: feed the GPU 64 images at a time.
* `.prefetch(AUTOTUNE)`: prepare the next batch while the GPU works on the current one.
* `.cache()`: **validation** (~7.5k images ≈ 1.1 GB as uint8) is cached in RAM. Training (≈ 10 GB decoded) and test (≈ 3.8 GB) are *not* cached in RAM. Use `TRAIN_CACHE_FILE` for an optional disk cache.
* **Shuffling happens after caching** (otherwise the cache would freeze the first epoch's order) and only for training.

**Common errors**
* *Slow epochs, GPU idle* → the CPU decode is the bottleneck; set `TRAIN_CACHE_FILE` so epochs 2+ skip JPEG decoding.
* *Colab RAM crash* → lower `SHUFFLE_BUFFER` or `BATCH_SIZE`.

In [ ]:
%%writefile food_inference.py
"""Shared preprocessing + inference code for the Food-101 MobileNetV2 classifier.

This file is the SINGLE SOURCE OF TRUTH for image preprocessing. The training
pipeline, the evaluation code, predict_food() and a future FastAPI service all
import `to_model_input` from here, so preprocessing cannot silently differ.
"""
import io
import json
import os

import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

IMG_SIZE = 224
ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif"}
ALLOWED_PIL_FORMATS = {"JPEG", "PNG", "BMP", "WEBP", "GIF"}
MAX_FILE_BYTES = 10 * 1024 * 1024
MIN_SIDE_PIXELS = 32
MODEL_FILENAME = "food_classifier_best.keras"

DISCLAIMER = ("Food-type classification from a photo only. It does not measure food "
              "quantity, freshness, remaining usable time, or food safety.")
CONFIDENCE_NOTE = ("Confidence is a raw softmax score, not a calibrated probability "
                   "that the prediction is correct.")


def to_model_input(image):
    """HxWx3 image (values 0-255, any dtype) -> uint8 tensor (IMG_SIZE, IMG_SIZE, 3).

    Steps: cast to float -> bilinear resize with antialiasing -> round -> clip -> uint8.
    The model itself rescales 0..255 -> -1..1 (MobileNetV2 convention) in its first layer.
    """
    image = tf.convert_to_tensor(image)
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear", antialias=True)
    image = tf.clip_by_value(tf.round(image), 0.0, 255.0)
    image = tf.cast(image, tf.uint8)
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return image


class ImageValidationError(Exception):
    """Raised for any input problem. `code` is machine-readable, `message` is safe to show."""

    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def decode_image_bytes(data, max_bytes=MAX_FILE_BYTES):
    """Validate raw bytes and return an RGB uint8 numpy array (H, W, 3)."""
    if not isinstance(data, (bytes, bytearray)) or len(data) == 0:
        raise ImageValidationError("EMPTY_FILE", "The file is empty.")
    if len(data) > max_bytes:
        raise ImageValidationError(
            "FILE_TOO_LARGE", "File exceeds the %d MB limit." % (max_bytes // (1024 * 1024)))
    try:
        with Image.open(io.BytesIO(data)) as probe:
            fmt = probe.format
            probe.verify()  # cheap integrity check; invalidates `probe`
        if fmt not in ALLOWED_PIL_FORMATS:
            raise ImageValidationError("UNSUPPORTED_FILE_TYPE", "Unsupported image format.")
        with Image.open(io.BytesIO(data)) as img:
            img = ImageOps.exif_transpose(img)  # honour phone-camera rotation
            img = img.convert("RGB")            # handles grayscale / RGBA / palette images
            arr = np.array(img, dtype=np.uint8)
    except ImageValidationError:
        raise
    except Exception:
        raise ImageValidationError("INVALID_IMAGE", "The file is not a readable image.") from None
    if arr.ndim != 3 or arr.shape[2] != 3 or min(arr.shape[:2]) < MIN_SIDE_PIXELS:
        raise ImageValidationError(
            "IMAGE_TOO_SMALL",
            "Image must be at least %dx%d pixels." % (MIN_SIDE_PIXELS, MIN_SIDE_PIXELS))
    return arr


def load_image_as_rgb_array(path, max_bytes=MAX_FILE_BYTES):
    """Validate a local file path and return an RGB uint8 numpy array."""
    try:
        path = os.fspath(path)
    except TypeError:
        raise ImageValidationError("INVALID_PATH", "Invalid file path.") from None
    if not os.path.isfile(path):
        raise ImageValidationError("FILE_NOT_FOUND", "File not found.")
    if os.path.splitext(path)[1].lower() not in ALLOWED_EXTENSIONS:
        raise ImageValidationError(
            "UNSUPPORTED_FILE_TYPE",
            "Unsupported file extension. Allowed: " + ", ".join(sorted(ALLOWED_EXTENSIONS)))
    try:
        with open(path, "rb") as f:
            data = f.read(max_bytes + 1)
    except OSError:
        raise ImageValidationError("FILE_UNREADABLE", "The file could not be read.") from None
    return decode_image_bytes(data, max_bytes)


def decide_uncertain(sorted_top_probs, conf_threshold, margin_threshold):
    """Return (needs_manual_verification, reasons) from the top probabilities (descending)."""
    top1 = float(sorted_top_probs[0])
    margin = top1 - float(sorted_top_probs[1]) if len(sorted_top_probs) > 1 else top1
    reasons = []
    if top1 < conf_threshold:
        reasons.append("low_confidence")
    if margin < margin_threshold:
        reasons.append("small_margin_between_top2")
    return len(reasons) > 0, reasons


def error_result(code, message):
    return {
        "status": "error",
        "error_code": code,
        "error_message": message,
        "needs_manual_verification": True,
    }


def apply_manual_correction(result, corrected_food, corrected_category=None):
    """Return a copy of a prediction result with the donor's correction recorded.

    `corrected_food` may be ANY text: the donor is not limited to the 101 Food-101 classes.
    """
    out = dict(result)
    out["user_correction"] = {
        "food": str(corrected_food).strip(),
        "category": corrected_category or result.get("food_category"),
    }
    out["needs_manual_verification"] = False
    out["verified_by_donor"] = True
    return out


class FoodRecognizer:
    """Wraps a trained Keras model + labels + category mapping + uncertainty rules."""

    def __init__(self, model, class_names, category_map=None, config=None,
                 conf_threshold=0.5, margin_threshold=0.10, top_k=3):
        n_out = int(model.output_shape[-1])
        if n_out != len(class_names):
            raise ValueError("Model has %d outputs but %d class names were given."
                             % (n_out, len(class_names)))
        self.model = model
        self.class_names = list(class_names)
        self.category_map = category_map if category_map is not None else {}
        self.config = config or {}
        self.conf_threshold = float(conf_threshold)
        self.margin_threshold = float(margin_threshold)
        self.top_k = min(int(top_k), len(self.class_names))

    @classmethod
    def from_dir(cls, artifact_dir, model_filename=MODEL_FILENAME):
        def _load(name, default):
            p = os.path.join(artifact_dir, name)
            if os.path.isfile(p):
                with open(p, "r", encoding="utf-8") as f:
                    return json.load(f)
            return default

        class_names = _load("class_labels.json", None)["classes"]
        category_map = _load("food_category_mapping.json", {})
        config = _load("model_config.json", {})
        unc = config.get("uncertainty", {})
        model = tf.keras.models.load_model(os.path.join(artifact_dir, model_filename),
                                           compile=False)
        return cls(model, class_names, category_map, config,
                   conf_threshold=unc.get("conf_threshold", 0.5),
                   margin_threshold=unc.get("margin_threshold", 0.10))

    # ---- public API -------------------------------------------------------
    def predict(self, image_path, conf_threshold=None, margin_threshold=None):
        return self._run(lambda: load_image_as_rgb_array(image_path),
                         conf_threshold, margin_threshold)

    def predict_bytes(self, data, conf_threshold=None, margin_threshold=None):
        """Same as predict() but for uploaded bytes (useful for a web API; no file path)."""
        return self._run(lambda: decode_image_bytes(data),
                         conf_threshold, margin_threshold)

    def predict_array(self, rgb_array, conf_threshold=None, margin_threshold=None):
        x = to_model_input(rgb_array)                     # SAME function as training
        x = tf.cast(x, tf.float32)[tf.newaxis, ...]
        probs = self.model(x, training=False).numpy()[0]
        return self._format(probs, conf_threshold, margin_threshold)

    # ---- internals --------------------------------------------------------
    def _run(self, loader, conf_threshold, margin_threshold):
        try:
            arr = loader()
        except ImageValidationError as e:
            return error_result(e.code, e.message)
        except Exception:
            return error_result("INTERNAL_ERROR", "The image could not be processed.")
        try:
            return self.predict_array(arr, conf_threshold, margin_threshold)
        except Exception:
            return error_result("INFERENCE_ERROR", "Prediction failed.")

    def _category(self, label):
        entry = self.category_map.get(label)
        if entry is None:
            return "unmapped", True
        return entry.get("category", "unmapped"), bool(entry.get("needs_review", True))

    def _format(self, probs, conf_threshold, margin_threshold):
        ct = self.conf_threshold if conf_threshold is None else float(conf_threshold)
        mt = self.margin_threshold if margin_threshold is None else float(margin_threshold)
        order = np.argsort(-probs)[: self.top_k]
        top_probs = [float(probs[i]) for i in order]
        uncertain, reasons = decide_uncertain(top_probs, ct, mt)
        best_label = self.class_names[int(order[0])]
        category, cat_review = self._category(best_label)
        return {
            "status": "ok",
            "predicted_label": best_label,
            "predicted_food": best_label.replace("_", " "),
            "food_category": category,
            "category_needs_review": cat_review,
            "confidence": round(top_probs[0], 4),
            "top_predictions": [
                {"label": self.class_names[int(i)],
                 "display_name": self.class_names[int(i)].replace("_", " "),
                 "confidence": round(float(probs[i]), 4)}
                for i in order
            ],
            "needs_manual_verification": bool(uncertain),
            "uncertainty_reasons": reasons,
            "confidence_note": CONFIDENCE_NOTE,
            "disclaimer": DISCLAIMER,
        }

In [ ]:
import importlib, sys
sys.path.insert(0, os.getcwd())
import food_inference
importlib.reload(food_inference)
from food_inference import to_model_input
assert food_inference.IMG_SIZE == IMG_SIZE

AUTOTUNE = tf.data.AUTOTUNE

def decode_example(ex):
    """raw TFDS example -> (uint8 image [224,224,3], int label)."""
    img = tf.io.decode_image(ex["image"], channels=3, expand_animations=False)  # forces RGB
    img.set_shape([None, None, 3])
    return to_model_input(img), ex["label"]

def to_float(x, y):
    return tf.cast(x, tf.float32), y   # model expects float32 pixel values in 0..255

def build_pipeline(raw_ds, training, cache=None):
    ds = raw_ds.map(decode_example, num_parallel_calls=AUTOTUNE)
    if cache == "memory":
        ds = ds.cache()
    elif cache:                                   # a file path => disk cache
        ds = ds.cache(cache)
    if training:
        ds = ds.shuffle(CFG["SHUFFLE_BUFFER"], seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(to_float, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = maybe_limit(build_pipeline(raw_train, training=True, cache=CFG["TRAIN_CACHE_FILE"]), "train")
val_ds = maybe_limit(build_pipeline(raw_val, training=False, cache="memory"), "val")
test_ds = maybe_limit(build_pipeline(raw_test, training=False, cache=None), "test")

# ---- look at one batch to confirm shapes, types and labels ----
xb, yb = next(iter(val_ds))
print("batch images:", xb.shape, xb.dtype, "| value range:", float(xb.numpy().min()), "-", float(xb.numpy().max()))
print("batch labels:", yb.shape, yb.dtype)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, im, lb in zip(axes.ravel(), xb.numpy(), yb.numpy()):
    ax.imshow(im.astype("uint8")); ax.axis("off"); ax.set_title(CLASS_NAMES[lb], fontsize=9)
plt.suptitle("Validation batch after preprocessing (224x224, no augmentation)"); plt.tight_layout(); plt.show()

# test-set order check (needed so labels can be matched to predictions later)
_, y_first = next(iter(test_ds))
assert np.array_equal(y_first.numpy(), test_labels_official[:len(y_first)]), "test order mismatch"
print("Test pipeline order check passed.")

## Step 7 · Data augmentation (training only)

**What it does:** randomly flips, rotates, zooms and changes the contrast of each training image, every epoch.
**Why:** the model sees slightly different versions of the same dish each time, which reduces overfitting and makes it more robust to camera angle and lighting.

**Parameters**
* `RandomFlip("horizontal")`: mirror left/right (food is not orientation-specific).
* `RandomRotation(0.05)`: up to ±5% of a full turn (±18°).
* `RandomZoom(0.1)`: zoom in/out by up to 10%.
* `RandomContrast(0.1)`: contrast change up to 10%.

**Why it is "training only":** Keras augmentation layers do nothing when the model runs in inference mode (`predict`, `evaluate`, `model(x, training=False)`). So validation, test and real predictions are never augmented, even though the layers are stored in the saved model.
**Order matters:** augmentation runs on raw 0–255 pixels, *before* the rescaling layer, because `RandomContrast` assumes a 0–255 range.

**Common errors:** *augmentation seems not to change images* → you must call it with `training=True` (as the demo below does).

In [ ]:
def build_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ], name="augmentation")

aug_demo = build_augmentation()
one = xb[:1]                                              # one validation image (float32, 0..255)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(one[0].numpy().astype("uint8")); axes[0].set_title("original"); axes[0].axis("off")
for ax in axes[1:]:
    aug = aug_demo(one, training=True)[0].numpy()
    ax.imshow(np.clip(aug, 0, 255).astype("uint8")); ax.set_title("augmented"); ax.axis("off")
plt.tight_layout(); plt.show()

same = aug_demo(one, training=False)[0].numpy()
print("Inference mode leaves the image unchanged:", np.allclose(same, one[0].numpy()))

## Step 8 · Model construction (MobileNetV2 transfer learning)

**In simple language**
MobileNetV2 was already trained on ImageNet (1.2M photos, 1,000 classes), so it already knows edges, textures, shapes and many food-like patterns. Instead of learning from zero, we **reuse** it:

```
image (224x224x3, 0..255)
  -> augmentation        (training only)
  -> rescale to [-1, 1]  (MobileNetV2 convention)
  -> MobileNetV2 backbone (ImageNet weights, FROZEN at first) -> 7x7x1280 feature map
  -> GlobalAveragePooling2D  -> 1280 numbers summarising the image
  -> Dropout(0.3)           -> randomly ignore 30% of features during training (reduces overfitting)
  -> Dense(101, softmax)    -> 101 scores that sum to 1
```

* `include_top=False` removes ImageNet's 1000-class head so we can attach our own 101-class head.
* **Freezing** (`trainable=False`) means only the small new head learns at first. That is fast and safe.
* Backbone is called with `training=False` so its BatchNorm layers keep their pretrained statistics.
* **Loss:** `sparse_categorical_crossentropy` because labels are integers (0–100), not one-hot vectors. **Optimizer:** Adam, learning rate 1e-3 for the new head.

**Common errors**
* *Cannot download ImageNet weights* → check internet; re-run.
* *Shape mismatch* → input must be 224×224×3.
* *Loss is `nan`* → learning rate too high or bad data; lower the LR.

In [ ]:
def build_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE, dropout=CFG["DROPOUT"], weights="imagenet"):
    inputs = keras.Input(shape=(img_size, img_size, 3), name="image")
    x = build_augmentation()(inputs)                                     # active only when training
    # x / 127.5 - 1  ==  keras.applications.mobilenet_v2.preprocess_input
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="preprocess_rescale")(x)
    backbone = keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3), include_top=False, weights=weights)
    backbone.trainable = False                                           # freeze for the first stage
    x = backbone(x, training=False)                                      # keep BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="food_probs")(x)
    return keras.Model(inputs, outputs, name="food101_mobilenetv2")

def get_backbone(m):
    for l in m.layers:
        if isinstance(l, keras.Model) and l.name.startswith("mobilenetv2"):
            return l
    raise ValueError("MobileNetV2 backbone layer not found")

def count_params(weights):
    return int(sum(np.prod(w.shape) for w in weights))

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(CFG["HEAD_LR"]),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
print("Trainable parameters (head only):", f"{count_params(model.trainable_weights):,}")
print("Frozen parameters (backbone):   ", f"{count_params(model.non_trainable_weights):,}")
print("Backbone layers:", len(get_backbone(model).layers))

## Step 9 · Initial training (classification head only)

**What it does:** trains only the new head while the MobileNetV2 backbone stays frozen.

**Plan (as you asked):** first a **short pipeline-check run** (`PIPELINE_CHECK_EPOCHS`, 2 epochs). If validation loss is still clearly improving, a second cell continues for a few more epochs. We do **not** assume any particular accuracy.

**Callbacks**
* `EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)`: stop when validation loss hasn't improved for 3 epochs and go back to the best weights.
* `ModelCheckpoint(save_best_only=True)`: writes a `.keras` file whenever validation loss improves. This is the "preserve the best checkpoint" safety net if Colab disconnects.
* `ReduceLROnPlateau`: halves the learning rate when validation loss stalls.

**How to tell if the pipeline works:** random guessing gives ≈ 1% accuracy (1/101). After the first epoch, validation accuracy should be **far** above 1%. If it stays near 1%, labels and images are misaligned or preprocessing is broken. Stop and debug before spending GPU time.

**Common errors**
* *OOM (out of memory)* → lower `BATCH_SIZE`.
* *Training much slower than expected* → see Step 6 (`TRAIN_CACHE_FILE`).
* *Train accuracy lower than validation accuracy early on* → normal: training accuracy is measured with augmentation and dropout switched on.

In [ ]:
BASELINE_CKPT = os.path.join(CKPT_DIR, "baseline_head_best.keras")
FINETUNED_CKPT = os.path.join(CKPT_DIR, "finetuned_best.keras")

def make_callbacks(ckpt_path, best_so_far=None):
    kw = {} if best_so_far is None else {"initial_value_threshold": best_so_far}
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=CFG["EARLY_STOP_PATIENCE"],
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True,
                                        verbose=1, **kw),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1,
                                          min_lr=1e-6, verbose=1),
    ]

head_histories = []
t0 = time.time()
h = model.fit(train_ds, validation_data=val_ds,
              epochs=CFG["PIPELINE_CHECK_EPOCHS"], callbacks=make_callbacks(BASELINE_CKPT), verbose=1)
head_histories.append(h.history)
print(f"\nPipeline-check run finished in {(time.time() - t0) / 60:.1f} min")
print(f"Validation accuracy after check run: {h.history['val_accuracy'][-1]:.3f}  (random guessing = {1 / NUM_CLASSES:.3f})")
if h.history["val_accuracy"][-1] < 5 * (1 / NUM_CLASSES) and not CFG["SMOKE_TEST"]:
    print("WARNING: accuracy is barely above chance. Check Steps 5-6 before training further.")

### Step 9b · Continue head training (only if it is still improving)

This cell applies a **simple rule of thumb** (not a statistical test): continue only if validation loss improved by more than 0.005 in the last epoch. It reloads the best checkpoint first, and the new `ModelCheckpoint` is told the best loss so far (`initial_value_threshold`), so a worse epoch can never overwrite a better checkpoint.

In [ ]:
def best_val_loss(histories):
    return min(min(hh["val_loss"]) for hh in histories)

last = head_histories[-1]["val_loss"]
improving = len(last) < 2 or (last[-2] - last[-1]) > 0.005
if CFG["EXTRA_HEAD_EPOCHS"] > 0 and improving:
    model = keras.models.load_model(BASELINE_CKPT)          # restart from the best checkpoint
    done = sum(len(hh["loss"]) for hh in head_histories)
    h = model.fit(train_ds, validation_data=val_ds, initial_epoch=done,
                  epochs=done + CFG["EXTRA_HEAD_EPOCHS"],
                  callbacks=make_callbacks(BASELINE_CKPT, best_so_far=best_val_loss(head_histories)),
                  verbose=1)
    head_histories.append(h.history)
else:
    print("Skipping extra head training (disabled, or validation loss is no longer improving clearly).")
print("Best validation loss so far (head training):", round(best_val_loss(head_histories), 4))

## Step 10 · Fine-tuning the upper MobileNetV2 layers

**What it does:** starts from the best head-only checkpoint, **unfreezes the upper part of the backbone** (layers with index ≥ `FINE_TUNE_AT`) and keeps training with a **much smaller learning rate** (1e-5).

**Why:** the early layers detect generic edges and textures that transfer well, but the later layers are specialised for ImageNet. Letting them adapt to food images usually improves accuracy.

**Important details**
* **Small LR:** with a large LR the pretrained weights would be destroyed ("catastrophic forgetting").
* **BatchNorm layers stay frozen:** fine-tuning BatchNorm with small batches can hurt. We also call the backbone with `training=False` in the model definition.
* **You must re-compile** after changing which layers are trainable. Otherwise the change is ignored.
* It uses its own checkpoint (`finetuned_best.keras`), so the baseline is never overwritten. Step 11 compares them fairly on the **validation** set.

**Common errors:** *validation loss gets worse immediately* → lower `FINE_TUNE_LR` (e.g. 3e-6) or raise `FINE_TUNE_AT` so fewer layers are unfrozen. *OOM* → lower `BATCH_SIZE`.

In [ ]:
model = keras.models.load_model(BASELINE_CKPT)       # start from the best head-only weights
backbone = get_backbone(model)
print("Backbone layers:", len(backbone.layers), "| unfreezing from layer index", CFG["FINE_TUNE_AT"])

backbone.trainable = True
for layer in backbone.layers[:CFG["FINE_TUNE_AT"]]:
    layer.trainable = False                          # keep early layers frozen
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False                      # keep BatchNorm statistics frozen

model.compile(optimizer=keras.optimizers.Adam(CFG["FINE_TUNE_LR"]),   # re-compile after changing trainable flags
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Trainable parameters now:", f"{count_params(model.trainable_weights):,}")

t0 = time.time()
ft_history = model.fit(train_ds, validation_data=val_ds, epochs=CFG["FINE_TUNE_EPOCHS"],
                       callbacks=make_callbacks(FINETUNED_CKPT), verbose=1).history
print(f"\nFine-tuning finished in {(time.time() - t0) / 60:.1f} min")

## Step 11 · Loading the checkpoints and choosing the best model

**What it does:** reloads the **best saved checkpoint** of each stage, evaluates both on the **validation** set, and keeps the one with the lower validation loss.

**Why validation and not test:** choosing a model by looking at test results would quietly leak the test set into your decisions and make the final number over-optimistic. The test set is used **once**, in Step 12, for the selected model only.

**Common errors:** *"No such file ... finetuned_best.keras"* → fine-tuning (Step 10) did not finish or the runtime was reset; re-run Step 10.

In [ ]:
def load_for_eval(path):
    m = keras.models.load_model(path)
    m.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

baseline_model = load_for_eval(BASELINE_CKPT)
ft_model = load_for_eval(FINETUNED_CKPT)

rows = []
for name, m in [("baseline (head only)", baseline_model), ("fine-tuned", ft_model)]:
    r = m.evaluate(val_ds, verbose=0, return_dict=True)
    rows.append({"model": name, "val_loss": r["loss"], "val_accuracy": r["accuracy"]})
selection_df = pd.DataFrame(rows)
display(selection_df)

if selection_df.loc[1, "val_loss"] < selection_df.loc[0, "val_loss"]:
    SELECTED_NAME, SELECTED_CKPT, best_model = "fine_tuned", FINETUNED_CKPT, ft_model
else:
    SELECTED_NAME, SELECTED_CKPT, best_model = "baseline_head_only", BASELINE_CKPT, baseline_model
print(f"\nSelected model (by validation loss): {SELECTED_NAME}")
print("Checkpoint:", SELECTED_CKPT)

## Step 12 · Final evaluation on the official held-out test set

**What it does:** evaluates the selected model on the official test split (25,250 images, or a small subset in smoke-test mode), **once**, and computes metrics from the **actual predictions**.

**Metrics explained**
* **Accuracy:** fraction of images whose top-1 prediction is correct.
* **Precision (per class):** of the images predicted as class *c*, how many really were *c*.
* **Recall (per class):** of the images that really are class *c*, how many were found.
* **F1:** harmonic mean of precision and recall.
* **Macro** averages treat all 101 classes equally; **weighted** averages weight by class size (identical here because the test set is balanced).
* **Top-3 accuracy:** true class is among the 3 best guesses. Relevant because your app shows 3 suggestions.

`model.evaluate` gives the loss/accuracy; `model.predict` gives probabilities for all images, from which scikit-learn computes everything else.

**Common errors:** *shape mismatch between `y_true` and `y_pred`* → the test pipeline must stay un-shuffled (it is).

In [ ]:
t0 = time.time()
test_eval = best_model.evaluate(test_ds, verbose=1, return_dict=True)
test_probs = best_model.predict(test_ds, verbose=1)
n_eval = len(test_probs)
y_true = test_labels_official[:n_eval]
y_pred = test_probs.argmax(axis=1)
print(f"Test evaluation took {(time.time() - t0) / 60:.1f} min on {n_eval} images")

labels_all = np.arange(NUM_CLASSES)
acc = float(accuracy_score(y_true, y_pred))
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels_all, average="macro", zero_division=0)
f1_weighted = float(f1_score(y_true, y_pred, labels=labels_all, average="weighted", zero_division=0))
top3_idx = np.argsort(-test_probs, axis=1)[:, :3]
top3_acc = float((top3_idx == y_true[:, None]).any(axis=1).mean())
manual_loss = float(-np.mean(np.log(np.clip(test_probs[np.arange(n_eval), y_true], 1e-12, 1.0))))

EVAL = {
    "evaluated_on": "official Food-101 test split (TFDS 'validation'), used once, after model selection",
    "n_test_images": int(n_eval),
    "smoke_test_run": bool(CFG["SMOKE_TEST"]),
    "selected_model": SELECTED_NAME,
    "test_loss": float(test_eval["loss"]),
    "test_loss_recomputed_from_predictions": manual_loss,
    "test_accuracy": float(test_eval["accuracy"]),
    "test_accuracy_recomputed_from_predictions": acc,
    "top3_accuracy": top3_acc,
    "macro_precision": float(p_macro),
    "macro_recall": float(r_macro),
    "macro_f1": float(f1_macro),
    "weighted_f1": f1_weighted,
}
if abs(EVAL["test_accuracy"] - acc) > 1e-3:
    print("NOTE: evaluate() and predict() accuracies differ slightly; check for nondeterminism.")

report_dict = classification_report(y_true, y_pred, labels=labels_all, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
report_text = classification_report(y_true, y_pred, labels=labels_all, target_names=CLASS_NAMES,
                                    zero_division=0, digits=3)
print("\n=== MEASURED TEST RESULTS ===" + ("   (SMOKE TEST: NOT MEANINGFUL)" if CFG["SMOKE_TEST"] else ""))
for k, v in EVAL.items():
    print(f"{k:45s} {v if not isinstance(v, float) else round(v, 4)}")

## Step 13 · Metrics report and graphs

**What you get**
1. The full **classification report** (precision / recall / F1 for every class).
2. Best and worst classes by F1.
3. **Training vs validation** accuracy and loss curves (the dashed line marks where fine-tuning starts).
4. The **101×101 confusion matrix** (rows = true class, columns = predicted class; bright off-diagonal cells are mistakes). With 101 classes it is dense, so we also show a **clearly labelled subset** (the 20 lowest-F1 classes) with counts, and a table of the most frequent confusions.

**How to read the curves**
* Validation loss rising while training loss keeps falling = **overfitting**.
* Training accuracy can sit *below* validation accuracy early on because training uses augmentation and dropout.
* Everything is saved as PNG in the artifacts folder. Use these figures in your report.

**Common errors:** *the full confusion-matrix figure looks tiny* → open the saved PNG (`plots/confusion_matrix_full.png`) and zoom in.

In [ ]:
print(report_text)

per_class = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": [report_dict[c]["precision"] for c in CLASS_NAMES],
    "recall": [report_dict[c]["recall"] for c in CLASS_NAMES],
    "f1": [report_dict[c]["f1-score"] for c in CLASS_NAMES],
    "support": [int(report_dict[c]["support"]) for c in CLASS_NAMES],
})
per_class = per_class[per_class["support"] > 0].sort_values("f1")
print("\n10 lowest-F1 classes:");  display(per_class.head(10).round(3))
print("10 highest-F1 classes:"); display(per_class.tail(10).iloc[::-1].round(3))
per_class.to_csv(os.path.join(ARTIFACT_DIR, "per_class_metrics.csv"), index=False)

In [ ]:
# ---------------- training vs validation curves ----------------
def merge_histories(hs):
    return {k: sum([hh[k] for hh in hs], []) for k in ["accuracy", "val_accuracy", "loss", "val_loss"]}

head = merge_histories(head_histories)
ft = {k: ft_history[k] for k in ["accuracy", "val_accuracy", "loss", "val_loss"]}
full = {k: head[k] + ft[k] for k in head}
n_head = len(head["loss"])
epochs_x = np.arange(1, len(full["loss"]) + 1)

for metric, title, fname in [("accuracy", "Accuracy", "accuracy_curves.png"), ("loss", "Loss", "loss_curves.png")]:
    plt.figure(figsize=(8, 4.5))
    plt.plot(epochs_x, full[metric], marker="o", label=f"training {metric}")
    plt.plot(epochs_x, full["val_" + metric], marker="o", label=f"validation {metric}")
    plt.axvline(n_head + 0.5, color="gray", linestyle="--", label="fine-tuning starts")
    plt.xlabel("epoch"); plt.ylabel(metric); plt.title(f"Training vs validation {title}")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fname), dpi=150); plt.show()

# ---------------- confusion matrix (full 101 x 101) ----------------
cm = confusion_matrix(y_true, y_pred, labels=labels_all)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(20, 18))
im = ax.imshow(cm_norm, cmap="viridis")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=5)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=5)
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_title("Confusion matrix, row-normalised (fraction of each true class)")
plt.colorbar(im, fraction=0.03); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "confusion_matrix_full.png"), dpi=150); plt.show()

# ---------------- readable subset: 20 lowest-F1 classes ----------------
f1_by_class = np.array([report_dict[c]["f1-score"] if report_dict[c]["support"] > 0 else 1.0 for c in CLASS_NAMES])
worst = np.argsort(f1_by_class)[:20]
sub = cm[np.ix_(worst, worst)]
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(sub, cmap="Blues")
names = [CLASS_NAMES[i] for i in worst]
ax.set_xticks(range(len(worst))); ax.set_xticklabels(names, rotation=90, fontsize=8)
ax.set_yticks(range(len(worst))); ax.set_yticklabels(names, fontsize=8)
for i in range(len(worst)):
    for j in range(len(worst)):
        if sub[i, j] > 0:
            ax.text(j, i, int(sub[i, j]), ha="center", va="center", fontsize=7,
                    color="white" if sub[i, j] > sub.max() / 2 else "black")
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_title("SUBSET: confusion counts among the 20 lowest-F1 classes\n(predictions outside these 20 classes are not shown)")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "confusion_matrix_subset_worst20.png"), dpi=150); plt.show()

# ---------------- most frequent confusions ----------------
off = cm.copy(); np.fill_diagonal(off, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:15], off.shape))[0]
confusions = pd.DataFrame([{"true_class": CLASS_NAMES[i], "predicted_as": CLASS_NAMES[j], "count": int(off[i, j])}
                           for i, j in top_pairs if off[i, j] > 0])
print("Most frequent confusions:"); display(confusions)

## Step 14 · Food-category mapping (configurable, with ambiguity flags)

**What it does:** maps each of the 101 Food-101 labels to a **broader food category** and stores it as an editable dictionary / JSON file.

**Why:** a donor listing needs a general type (e.g. *soup*, *dessert_sweet*) as well as a specific dish name. These categories are **my draft suggestions**, not an official taxonomy. **Review all of them** against your project's needs.

**Honest caveats**
* The categories describe **what kind of dish it is**, *not* donation suitability, storage needs, perishability or safety. (For example, `ice_cream` and `sushi` have very different handling requirements. Your system must get that information from the donor, not from this classifier.)
* Labels that could reasonably fit more than one category are flagged `needs_review: true` with a note, so the app can ask for confirmation.
* Any label without a mapping becomes `"unmapped"` + `needs_review`.
* To correct a mapping, call `update_category_mapping("sushi", "rice_dish")`. It clears the review flag and re-saves the JSON.

**Common errors:** *KeyError / label not found* → use the exact Food-101 label (lowercase with underscores, e.g. `french_fries`).

In [ ]:
CATEGORY_GROUPS = {
    "dessert_sweet": ["apple_pie", "baklava", "beignets", "bread_pudding", "cannoli", "carrot_cake",
                      "cheesecake", "chocolate_cake", "chocolate_mousse", "churros", "creme_brulee",
                      "cup_cakes", "donuts", "frozen_yogurt", "ice_cream", "macarons", "panna_cotta",
                      "red_velvet_cake", "strawberry_shortcake", "tiramisu"],
    "breakfast_item": ["french_toast", "pancakes", "waffles"],
    "egg_dish": ["deviled_eggs", "eggs_benedict", "omelette", "huevos_rancheros"],
    "meat_dish": ["baby_back_ribs", "beef_carpaccio", "beef_tartare", "chicken_wings", "filet_mignon",
                  "foie_gras", "peking_duck", "pork_chop", "prime_rib", "steak"],
    "seafood_dish": ["ceviche", "crab_cakes", "escargots", "fish_and_chips", "fried_calamari",
                     "grilled_salmon", "mussels", "oysters", "sashimi", "scallops", "shrimp_and_grits",
                     "tuna_tartare"],
    "salad": ["beet_salad", "caesar_salad", "caprese_salad", "greek_salad", "seaweed_salad"],
    "soup": ["clam_chowder", "french_onion_soup", "hot_and_sour_soup", "lobster_bisque", "miso_soup", "pho"],
    "pasta_noodles": ["gnocchi", "lasagna", "macaroni_and_cheese", "pad_thai", "ramen", "ravioli",
                      "spaghetti_bolognese", "spaghetti_carbonara"],
    "rice_dish": ["bibimbap", "fried_rice", "paella", "risotto"],
    "sandwich_burger_wrap": ["breakfast_burrito", "chicken_quesadilla", "club_sandwich", "croque_madame",
                             "grilled_cheese_sandwich", "hamburger", "hot_dog", "lobster_roll_sandwich",
                             "pulled_pork_sandwich", "tacos"],
    "pizza": ["pizza"],
    "bread_bakery": ["bruschetta", "garlic_bread"],
    "appetizer_snack_side": ["cheese_plate", "dumplings", "edamame", "falafel", "french_fries", "guacamole",
                             "gyoza", "hummus", "nachos", "onion_rings", "poutine", "samosa",
                             "spring_rolls", "takoyaki"],
    "prepared_dish": ["chicken_curry", "sushi"],
}

# labels where the category is debatable -> flagged for human review
AMBIGUOUS = {
    "sushi": "rice + raw fish; could be rice_dish or seafood_dish",
    "pho": "noodle soup; could be soup or pasta_noodles",
    "ramen": "noodle soup; could be soup or pasta_noodles",
    "escargots": "snails; not clearly meat or seafood",
    "foie_gras": "organ meat; may be a spread/appetizer",
    "fish_and_chips": "fried fish plus fries; seafood or fried meal",
    "shrimp_and_grits": "seafood on a grain/porridge base",
    "french_toast": "breakfast or dessert", "pancakes": "breakfast or dessert", "waffles": "breakfast or dessert",
    "croque_madame": "sandwich with egg",
    "huevos_rancheros": "egg dish served on tortilla",
    "poutine": "fries with cheese and gravy; side or full meal",
    "nachos": "snack or full meal",
    "tacos": "wrap-style or separate dish",
    "dumplings": "snack, side or main", "gyoza": "snack, side or main",
    "samosa": "snack or main", "spring_rolls": "snack or starter", "takoyaki": "snack or main",
    "cheese_plate": "dairy platter; may be mixed foods",
    "bruschetta": "bread-based appetizer", "garlic_bread": "bread or side dish",
    "chicken_curry": "generic curry; depends on the dish served with it",
    "bibimbap": "rice bowl with many toppings (mixed dish)",
    "paella": "rice dish with mixed meat/seafood",
    "macaroni_and_cheese": "pasta or comfort-food main",
    "baklava": "pastry or dessert", "bread_pudding": "baked good or dessert",
    "guacamole": "dip; vegetable-based", "hummus": "dip; legume-based", "falafel": "snack or main",
    "edamame": "vegetable snack",
    "lobster_bisque": "soup made from seafood",
}

CATEGORY_MAP = {}
for cat, labs in CATEGORY_GROUPS.items():
    for lab in labs:
        if lab in CATEGORY_MAP:
            warnings.warn(f"{lab} appears in more than one category group")
        CATEGORY_MAP[lab] = {"category": cat, "needs_review": lab in AMBIGUOUS, "note": AMBIGUOUS.get(lab, "")}

unmapped = [c for c in CLASS_NAMES if c not in CATEGORY_MAP]
unknown_keys = [k for k in CATEGORY_MAP if k not in CLASS_NAMES]
for c in unmapped:
    CATEGORY_MAP[c] = {"category": "unmapped", "needs_review": True, "note": "no mapping defined yet"}
if unmapped:    print("Unmapped labels (set to 'unmapped'):", unmapped)
if unknown_keys: print("WARNING: mapping keys that are not Food-101 labels:", unknown_keys)
CATEGORY_MAP = {c: CATEGORY_MAP[c] for c in CLASS_NAMES}          # keep Food-101 label order

CATEGORY_JSON_PATH = os.path.join(ARTIFACT_DIR, "food_category_mapping.json")

def save_category_mapping(path=CATEGORY_JSON_PATH):
    save_json(CATEGORY_MAP, path)

def update_category_mapping(label, new_category, note="manually reviewed"):
    """Manually correct one mapping and save the JSON again."""
    if label not in CATEGORY_MAP:
        raise KeyError(f"'{label}' is not a Food-101 label")
    CATEGORY_MAP[label] = {"category": new_category, "needs_review": False, "note": note}
    save_category_mapping()
    print(f"Updated: {label} -> {new_category}")

save_category_mapping()
cat_df = pd.DataFrame([{"label": k, **v} for k, v in CATEGORY_MAP.items()])
print(f"{len(cat_df)} labels mapped into {cat_df['category'].nunique()} categories; "
      f"{int(cat_df['needs_review'].sum())} flagged for review.\n")
display(cat_df.groupby("category").agg(n_labels=("label", "count"), n_need_review=("needs_review", "sum")))
print("Flagged for review:"); display(cat_df[cat_df["needs_review"]][["label", "category", "note"]])

## Step 15 · Prediction function, uncertainty handling and manual correction

### 15a · Choosing the "uncertain" rule (using validation data only)
A prediction is flagged **`needs_manual_verification`** when
1. top-1 confidence `< CONF_THRESHOLD`, **or**
2. the gap between the top-1 and top-2 scores `< MARGIN_THRESHOLD` (two candidates are nearly tied).

`CONF_THRESHOLD` is chosen on the **validation** set: the smallest threshold at which predictions *above* it reach `TARGET_ACCEPTED_ACC` (90%) accuracy. This target and the margin (0.10) are **design choices you can change**, not scientific constants.

**Limitations of confidence thresholds (important):**
* Softmax scores are **not calibrated probabilities**. "0.9" does not mean "90% likely correct", and neural networks are often over-confident.
* The model must pick one of 101 classes **even for non-food or out-of-vocabulary images** (a dish with no Food-101 class). It can still be confidently wrong. A threshold cannot catch every error.
* The threshold trades coverage against accuracy. A high threshold flags more images (more manual work), a low one lets more mistakes through.
* Photos that differ from Food-101 (bulk food, containers, poor light) tend to produce lower scores, which is useful, but not guaranteed.

Hence the design: the model only **suggests**; the **donor can always correct** the name and category.

In [ ]:
# ---- thresholds chosen on the VALIDATION set ----
val_probs = best_model.predict(val_ds, verbose=0)
val_true = np.concatenate([y.numpy() for _, y in val_ds])
val_pred = val_probs.argmax(axis=1)
srt_v = np.sort(val_probs, axis=1)
v_top1, v_margin = srt_v[:, -1], srt_v[:, -1] - srt_v[:, -2]
MARGIN_THRESHOLD = CFG["MARGIN_THRESHOLD"]

rows = []
for t in np.round(np.arange(0.10, 0.96, 0.05), 2):
    accepted = (v_top1 >= t) & (v_margin >= MARGIN_THRESHOLD)
    rows.append({"conf_threshold": t,
                 "coverage (share NOT flagged)": float(accepted.mean()),
                 "accuracy_of_unflagged": float((val_pred[accepted] == val_true[accepted]).mean()) if accepted.any() else np.nan})
THRESH_TABLE = pd.DataFrame(rows)
display(THRESH_TABLE.round(3))

ok = THRESH_TABLE[THRESH_TABLE["accuracy_of_unflagged"] >= CFG["TARGET_ACCEPTED_ACC"]]
if len(ok):
    CONF_THRESHOLD = float(ok.iloc[0]["conf_threshold"]); THRESH_HOW = "smallest validation threshold reaching target accuracy"
else:
    CONF_THRESHOLD = 0.50; THRESH_HOW = "FALLBACK default 0.50: target accuracy not reached on validation data"
    print("WARNING: no threshold reached the target accuracy; using fallback 0.50.")
print(f"\nCONF_THRESHOLD = {CONF_THRESHOLD}  ({THRESH_HOW}) | MARGIN_THRESHOLD = {MARGIN_THRESHOLD}")

# ---- honest report of how this rule behaves on the TEST set (not used to choose it) ----
srt_t = np.sort(test_probs, axis=1)
t_top1, t_margin = srt_t[:, -1], srt_t[:, -1] - srt_t[:, -2]
flag_t = (t_top1 < CONF_THRESHOLD) | (t_margin < MARGIN_THRESHOLD)
correct_t = (y_pred == y_true)
def _safe_mean(a): return float(np.mean(a)) if len(a) else float("nan")
UNCERTAINTY_ON_TEST = {
    "conf_threshold": CONF_THRESHOLD, "margin_threshold": MARGIN_THRESHOLD, "how_chosen": THRESH_HOW,
    "fraction_flagged": float(flag_t.mean()),
    "accuracy_when_not_flagged": _safe_mean(correct_t[~flag_t]),
    "accuracy_when_flagged": _safe_mean(correct_t[flag_t]),
    "mean_top1_confidence_when_correct": _safe_mean(t_top1[correct_t]),
    "mean_top1_confidence_when_wrong": _safe_mean(t_top1[~correct_t]),
}
EVAL["uncertainty_rule_on_test"] = UNCERTAINTY_ON_TEST
for k, v in UNCERTAINTY_ON_TEST.items():
    print(f"{k:40s} {round(v, 4) if isinstance(v, float) else v}")

### 15b · `predict_food(image_path)`

`predict_food()` is a thin wrapper around `FoodRecognizer` (in `food_inference.py`). It:
1. checks the path, file extension, file size and that the file is a real image (Pillow `verify()`),
2. applies the phone-camera rotation (EXIF), converts to **RGB**,
3. resizes and preprocesses with the **same `to_model_input()` used in training**,
4. runs the model, returns the **top-3** labels and confidences, the category, and the uncertainty flag,
5. **never raises** for bad input. It returns `{"status": "error", "error_code": ..., ...}` with a safe message that never contains file paths or stack traces.

Note: training decodes JPEGs with TensorFlow and inference decodes with Pillow. These can differ by a pixel level or two; the tests in Step 19 measure this. The EXIF rotation step is applied only at inference (Food-101 images are already upright).

**Manual correction:** `apply_manual_correction(result, "dal makhani", "prepared_dish")` records what the donor says. The donor may type **any** food name, not only the 101 classes. Saving such corrections is a good way to collect data for future retraining (not implemented here).

**Common errors:** `FILE_NOT_FOUND` → check the path (Colab's file panel → "Copy path"); `UNSUPPORTED_FILE_TYPE` → use jpg/png/bmp/webp/gif; `INVALID_IMAGE` → the file is corrupt or not an image.

In [ ]:
from food_inference import FoodRecognizer, apply_manual_correction

recognizer = FoodRecognizer(best_model, CLASS_NAMES, CATEGORY_MAP, config={},
                            conf_threshold=CONF_THRESHOLD, margin_threshold=MARGIN_THRESHOLD)

def predict_food(image_path):
    """Return a JSON-compatible dict with top-3 predictions, category and uncertainty flag."""
    return recognizer.predict(image_path)

# ---- save some REAL official-test images to disk (used for demos and for the software tests) ----
lowest_i, highest_i = int(np.argmin(t_top1)), int(np.argmax(t_top1))
wanted = {i: "first" for i in range(5) if i < n_eval}
wanted[lowest_i] = "lowconf"       # the test image the model was LEAST sure about
wanted[highest_i] = "highconf"     # the test image the model was MOST sure about
mask = np.zeros(len(test_labels_official), dtype=bool); mask[list(wanted)] = True
SAMPLE_IMAGES = []
for idx, ex in zip(sorted(wanted), tfds.as_numpy(subset_by_mask(raw_test, mask))):
    path = os.path.join(SAMPLE_DIR, f"{wanted[idx]}_{idx:05d}_{CLASS_NAMES[ex['label']]}.jpg")
    with open(path, "wb") as f:
        f.write(ex["image"])                       # original JPEG bytes
    SAMPLE_IMAGES.append({"path": path, "true_label": CLASS_NAMES[ex["label"]], "tag": wanted[idx]})
print(f"Saved {len(SAMPLE_IMAGES)} real test images to {SAMPLE_DIR}")

def show_prediction(image_path, result):
    img = Image.open(image_path).convert("RGB")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.2]})
    axes[0].imshow(img); axes[0].axis("off")
    if result.get("status") == "ok":
        tp = result["top_predictions"][::-1]
        axes[1].barh([t["display_name"] for t in tp], [t["confidence"] for t in tp], color="tab:blue")
        axes[1].set_xlim(0, 1); axes[1].set_xlabel("softmax confidence (NOT a calibrated probability)")
        flag = "  ⚠ needs manual verification" if result["needs_manual_verification"] else ""
        axes[0].set_title(f"{result['predicted_food']} [{result['food_category']}]{flag}", fontsize=10)
    else:
        axes[1].text(0.5, 0.5, result.get("error_message", "error"), ha="center"); axes[1].axis("off")
    plt.tight_layout(); plt.show()

for s in SAMPLE_IMAGES[:3]:
    res = predict_food(s["path"])
    print("TRUE label (from dataset):", s["true_label"])
    print(json.dumps(res, indent=2))
    show_prediction(s["path"], res)

# ---- an invalid input: a safe error, no crash ----
print(json.dumps(predict_food("/content/does_not_exist.jpg"), indent=2))

In [ ]:
# ---------------- manual verification / correction ----------------
def review_prediction(result, ask=input):
    """Let the donor accept or correct the model's suggestion."""
    if result.get("status") != "ok":
        print("Cannot review:", result.get("error_message")); return result
    print(f"Model suggests: {result['predicted_food']}  (category: {result['food_category']}, "
          f"confidence {result['confidence']:.2f})")
    if result["needs_manual_verification"]:
        print("The model is UNSURE. Please check carefully. Top-3:",
              [t["display_name"] for t in result["top_predictions"]])
    name = ask("Press Enter to accept, or type the correct food name: ").strip()
    if not name:
        out = dict(result); out["needs_manual_verification"] = False; out["verified_by_donor"] = True
        return out
    cat = ask(f"Category (Enter = keep '{result['food_category']}'): ").strip() or None
    return apply_manual_correction(result, name, cat)

INTERACTIVE_REVIEW = False      # True = you type answers yourself
sample_result = predict_food(SAMPLE_IMAGES[0]["path"])
if INTERACTIVE_REVIEW:
    final_listing = review_prediction(sample_result)
else:
    print("--- SIMULATED donor input (the answers below are scripted, not real) ---")
    scripted = iter(["homemade vegetable pulao", "rice_dish"])
    final_listing = review_prediction(sample_result, ask=lambda prompt: (print(prompt, end=""), next(scripted))[1])
print(json.dumps(final_listing, indent=2))

# Example of correcting the category mapping itself (uncomment to use):
# update_category_mapping("sushi", "rice_dish")

### 15c · Upload and test your own food image (Colab)

Set `UPLOAD_NOW = True`, run the cell, and pick a photo from your computer. (It defaults to `False` so that "Run all" doesn't wait for an upload.) Try different things: a pizza, a plate of biryani, a non-food object. Notice how the model behaves on foods **outside** the 101 classes.

In [ ]:
UPLOAD_NOW = False      # <-- set True, then run this cell to upload a photo

if UPLOAD_NOW:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        res = predict_food(fname)
        print(json.dumps(res, indent=2))
        show_prediction(fname, res)
else:
    print("Set UPLOAD_NOW = True and re-run this cell to upload an image.")

## Step 16 · Saving the model and all artifacts

**Files written to `/content/food_model_artifacts/`**
| File | Purpose |
|---|---|
| `food_classifier_best.keras` | The selected model (architecture + weights, including the built-in rescaling layer) |
| `class_labels.json` | Index → Food-101 label (order is critical: output neuron *i* = `classes[i]`) |
| `food_category_mapping.json` | Label → category (+ review flags) |
| `model_config.json` | Hyper-parameters, preprocessing description, uncertainty thresholds, library versions, limitations |
| `evaluation_metrics.json`, `classification_report.json`, `per_class_metrics.csv` | **Measured** results |
| `training_history.json`, `plots/` | Curves and confusion matrices |
| `food_inference.py` | Standalone preprocessing + prediction code (needed to reload and for a future API) |

After saving, the cell **re-loads the saved file** and checks that its predictions match the in-memory model.

**Common errors:** *"Unable to synchronously create file"* → the folder doesn't exist (re-run Step 2); *reload mismatch* → don't edit the model between saving and checking.

In [ ]:
final_model_path = os.path.join(ARTIFACT_DIR, "food_classifier_best.keras")
best_model.save(final_model_path)

save_json({"num_classes": NUM_CLASSES, "classes": CLASS_NAMES,
           "index_to_label": {str(i): n for i, n in enumerate(CLASS_NAMES)}},
          os.path.join(ARTIFACT_DIR, "class_labels.json"))
save_category_mapping()

model_config = {
    "project": "Dynamic AI-Assisted Surplus Food Redistribution Under Time and Demand Uncertainty",
    "task": "food-type image classification (Food-101, 101 classes)",
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "smoke_test_run": bool(CFG["SMOKE_TEST"]),
    "selected_model": SELECTED_NAME,
    "architecture": "MobileNetV2 (ImageNet weights) + GlobalAveragePooling2D + Dropout + Dense(101, softmax)",
    "input": {"shape": [IMG_SIZE, IMG_SIZE, 3], "dtype": "float32 pixel values in 0..255 (rescaling to [-1,1] is inside the model)"},
    "preprocessing": ("RGB -> resize 224x224 bilinear+antialias -> round -> uint8 (food_inference.to_model_input); "
                      "model layer 'preprocess_rescale' = x/127.5 - 1"),
    "hyperparameters": {k: v for k, v in CFG.items() if k != "SMOKE_BATCHES"},
    "dataset": {"name": "Food-101", "source": "https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/",
                "loaded_via": "tensorflow_datasets 'food101'",
                "splits": {"train": int(train_mask.sum()), "validation": int(val_mask.sum()),
                           "test_official": int(len(test_labels_official))},
                "note": "TFDS 'validation' split == official Food-101 test set"},
    "uncertainty": {"conf_threshold": CONF_THRESHOLD, "margin_threshold": MARGIN_THRESHOLD,
                    "how_chosen": THRESH_HOW,
                    "rule": "needs_manual_verification if top1 < conf_threshold OR (top1 - top2) < margin_threshold"},
    "versions": {"tensorflow": tf.__version__, "keras": keras.__version__, "tfds": tfds.__version__,
                 "numpy": np.__version__, "python": sys.version.split()[0]},
    "limitations": [
        "Predicts food TYPE only (one of 101 classes). It does NOT assess quantity, freshness, remaining usable time or safety.",
        "Food-101 images are mostly single plated dishes; real donation photos will differ, so expect lower accuracy.",
        "Only 101 classes; many regional dishes have no class and will be mapped to the nearest wrong class.",
        "Softmax confidence is not a calibrated probability; thresholds reduce but do not remove confident errors.",
        "Category mapping is a draft; ambiguous entries are flagged needs_review.",
    ],
}
save_json(model_config, os.path.join(ARTIFACT_DIR, "model_config.json"))
save_json({"test": EVAL, "validation_model_selection": selection_df.to_dict("records"),
           "validation_threshold_table": THRESH_TABLE.to_dict("records")},
          os.path.join(ARTIFACT_DIR, "evaluation_metrics.json"))
save_json(report_dict, os.path.join(ARTIFACT_DIR, "classification_report.json"))
save_json({"head_training": head_histories, "fine_tuning": ft_history},
          os.path.join(ARTIFACT_DIR, "training_history.json"))
shutil.copy(food_inference.__file__, os.path.join(ARTIFACT_DIR, "food_inference.py"))

# ---- verify the SAVED file reproduces the in-memory model ----
reloaded = keras.models.load_model(final_model_path, compile=False)
a = best_model(xb[:16], training=False).numpy()
b = reloaded(xb[:16], training=False).numpy()
print("Reloaded model matches in-memory model:", bool(np.allclose(a, b, atol=1e-4)))
assert np.allclose(a, b, atol=1e-4), "Saved model does not reproduce predictions!"

print("\nArtifacts in", ARTIFACT_DIR)
for root, _, fs in os.walk(ARTIFACT_DIR):
    for fn in sorted(fs):
        p = os.path.join(root, fn)
        print(f"  {os.path.relpath(p, ARTIFACT_DIR):50s} {os.path.getsize(p) / 1024:10.1f} KB")

## Step 17 · Optional: save the artifacts to Google Drive

Colab's disk is **deleted when the session ends**. Copying the artifacts to Drive keeps them. Set `CFG["SAVE_TO_DRIVE"] = False` in Step 2 to skip this step. Running it opens a pop-up asking you to authorise Drive access.

**Common errors:** *Drive mount fails* → re-run and allow all permissions; *"Not running in Colab"* → you are running somewhere else; copy `food_model_artifacts/` manually.

In [ ]:
if CFG["SAVE_TO_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copytree(ARTIFACT_DIR, CFG["DRIVE_FOLDER"], dirs_exist_ok=True)
        print("Copied artifacts to Google Drive:", CFG["DRIVE_FOLDER"])
    except ImportError:
        print("Not running in Google Colab, so Drive is unavailable. Copy", ARTIFACT_DIR, "manually.")
else:
    print("Drive export skipped (CFG['SAVE_TO_DRIVE'] is False).")

## Step 18 · Reload the saved model and run inference (works in a fresh Colab session)

**To use this in a brand-new session**
1. Mount Drive and set `ARTIFACT_DIR_RELOAD` to your Drive folder (see the commented lines).
2. Run **only this cell** (and the Step 19 tests if you like). It needs no training variables: everything comes from the saved files (`food_inference.py` is stored beside the model).
3. Point `IMAGE_PATH` to any food photo.

**Common errors:** *`ModuleNotFoundError: food_inference`* → wrong folder; *`No such file ... food_classifier_best.keras`* → artifacts weren't copied.

In [ ]:
import os, sys, json, glob

# --- In a FRESH session, uncomment these two lines and use your Drive folder: ---
# from google.colab import drive; drive.mount("/content/drive")
# ARTIFACT_DIR_RELOAD = "/content/drive/MyDrive/food_recognition_artifacts"
ARTIFACT_DIR_RELOAD = globals().get("ARTIFACT_DIR_RELOAD", "/content/food_model_artifacts")

sys.path.insert(0, ARTIFACT_DIR_RELOAD)
from food_inference import FoodRecognizer

rec_loaded = FoodRecognizer.from_dir(ARTIFACT_DIR_RELOAD)
print("Loaded model with", rec_loaded.model.output_shape[-1], "output classes.")
print("Uncertainty rule: conf <", rec_loaded.conf_threshold, "or margin <", rec_loaded.margin_threshold)

IMAGE_PATH = None        # <-- put the path of your own food photo here, e.g. "/content/my_food.jpg"
if IMAGE_PATH is None:
    found = sorted(glob.glob("/content/sample_test_images/first_*.jpg"))
    IMAGE_PATH = found[0] if found else None

if IMAGE_PATH:
    print(json.dumps(rec_loaded.predict(IMAGE_PATH), indent=2))
else:
    print("Set IMAGE_PATH to a food photo to test inference.")

## Step 19 · Software tests (NOT model-accuracy evaluation)

These tests check that the **code behaves correctly**: valid and invalid inputs, output structure, class count, the uncertainty logic and preprocessing consistency. A passing test does **not** say the model is accurate; accuracy is measured only by the test-set evaluation in Steps 12–13.

They run against `rec_loaded` (the model reloaded from the saved files) and use **real official test images** saved in Step 15, plus a few deliberately broken files. Tests that need sample images are skipped in a fresh session where those images don't exist.

| Test | What it proves |
|---|---|
| 01 | A real food image returns `status: ok` with a valid label |
| 02 | Output has exactly the documented top-3 structure, sorted, JSON-serialisable |
| 03 | Model/labels/category map all have 101 classes; softmax sums to 1 |
| 04 | Missing, empty, fake, oversized, tiny, truncated and wrong-type files fail **safely** with no path leak |
| 05 | Low-confidence logic: unit checks, threshold overrides, and the real least-confident test image |
| 06 | Preprocessing is reproducible; matches official MobileNetV2 preprocessing; grayscale/RGBA inputs work |

In [ ]:
import glob, tempfile, unittest
from PIL import Image
from tensorflow import keras
from food_inference import (to_model_input, load_image_as_rgb_array, decide_uncertain, MAX_FILE_BYTES)

TMP = tempfile.mkdtemp(prefix="food_tests_")
TEST_LOG = []
sample_paths = sorted(glob.glob("/content/sample_test_images/*.jpg"))

def need_samples():
    if not sample_paths:
        raise unittest.SkipTest("no sample images (run Step 15 first)")

def software_test(fn):
    try:
        fn(); TEST_LOG.append({"test": fn.__name__, "result": "PASS", "detail": ""})
    except unittest.SkipTest as e:
        TEST_LOG.append({"test": fn.__name__, "result": "SKIP", "detail": str(e)})
    except AssertionError as e:
        TEST_LOG.append({"test": fn.__name__, "result": "FAIL", "detail": str(e)[:300]})
    except Exception as e:
        TEST_LOG.append({"test": fn.__name__, "result": "ERROR", "detail": repr(e)[:300]})
    return fn

@software_test
def test_01_valid_food_image_returns_ok():
    need_samples()
    for p in sample_paths[:3]:
        r = rec_loaded.predict(p)
        assert r["status"] == "ok", r
        assert r["predicted_label"] in rec_loaded.class_names

@software_test
def test_02_top3_output_structure():
    need_samples()
    r = rec_loaded.predict(sample_paths[0])
    required = {"status", "predicted_label", "predicted_food", "food_category", "confidence",
                "top_predictions", "needs_manual_verification"}
    assert required <= set(r), f"missing keys: {required - set(r)}"
    tp = r["top_predictions"]
    assert len(tp) == 3, "expected exactly 3 predictions"
    confs = [t["confidence"] for t in tp]
    assert confs == sorted(confs, reverse=True), "top-3 not sorted"
    assert all(0.0 <= c <= 1.0 for c in confs) and sum(confs) <= 1.0005
    assert len({t["label"] for t in tp}) == 3, "duplicate labels in top-3"
    assert r["confidence"] == confs[0] and r["predicted_label"] == tp[0]["label"]
    assert isinstance(r["needs_manual_verification"], bool)
    json.loads(json.dumps(r))                     # must be JSON-compatible

@software_test
def test_03_number_of_output_classes():
    assert rec_loaded.model.output_shape[-1] == 101
    assert len(rec_loaded.class_names) == 101 == len(set(rec_loaded.class_names))
    assert set(rec_loaded.category_map) == set(rec_loaded.class_names), "category map keys != class names"
    x = np.random.RandomState(0).randint(0, 256, (4, 224, 224, 3)).astype("float32")
    p = rec_loaded.model(x, training=False).numpy()
    assert p.shape == (4, 101) and np.allclose(p.sum(axis=1), 1.0, atol=1e-4)

@software_test
def test_04_invalid_inputs_fail_safely():
    txt = os.path.join(TMP, "fake_image.jpg");  open(txt, "w").write("this is not an image")
    empty = os.path.join(TMP, "empty.jpg");     open(empty, "wb").close()
    wrong_ext = os.path.join(TMP, "notes.txt"); open(wrong_ext, "w").write("hello")
    big = os.path.join(TMP, "big.jpg");         open(big, "wb").write(b"\xff\xd8" + b"0" * (MAX_FILE_BYTES + 10))
    tiny = os.path.join(TMP, "tiny.png");       Image.new("RGB", (8, 8)).save(tiny)
    cases = [(os.path.join(TMP, "missing.jpg"), "FILE_NOT_FOUND"), (TMP, "FILE_NOT_FOUND"),
             (txt, "INVALID_IMAGE"), (empty, "EMPTY_FILE"), (wrong_ext, "UNSUPPORTED_FILE_TYPE"),
             (big, "FILE_TOO_LARGE"), (tiny, "IMAGE_TOO_SMALL"), (None, "INVALID_PATH"), (12345, "INVALID_PATH")]
    if sample_paths:
        trunc = os.path.join(TMP, "truncated.jpg")
        open(trunc, "wb").write(open(sample_paths[0], "rb").read()[:600])
        cases.append((trunc, "INVALID_IMAGE"))
    for path, expected in cases:
        r = rec_loaded.predict(path)
        assert r["status"] == "error" and r["error_code"] == expected, (path, expected, r)
        assert r["needs_manual_verification"] is True
        blob = json.dumps(r)
        assert TMP not in blob and "Traceback" not in blob, "error leaked internal details"
    r = rec_loaded.predict_bytes(b"garbage bytes")
    assert r["status"] == "error" and r["error_code"] == "INVALID_IMAGE"

@software_test
def test_05_low_confidence_flag_logic():
    # pure logic (no model involved)
    assert decide_uncertain([0.95, 0.03, 0.01], 0.5, 0.1) == (False, [])
    assert decide_uncertain([0.30, 0.25, 0.20], 0.5, 0.1)[0] is True
    flag, reasons = decide_uncertain([0.55, 0.50, 0.10], 0.5, 0.1)
    assert flag and reasons == ["small_margin_between_top2"]
    need_samples()
    p = sample_paths[0]
    assert rec_loaded.predict(p, conf_threshold=1.01)["needs_manual_verification"] is True    # everything flagged
    assert rec_loaded.predict(p, conf_threshold=0.0, margin_threshold=0.0)["needs_manual_verification"] is False
    low = [q for q in sample_paths if os.path.basename(q).startswith("lowconf_")]
    if low:                                      # the real test image the model was least sure about
        r = rec_loaded.predict(low[0])
        expected, _ = decide_uncertain([t["confidence"] for t in r["top_predictions"]],
                                       rec_loaded.conf_threshold, rec_loaded.margin_threshold)
        assert r["needs_manual_verification"] == expected
        print(f"   least-confident real test image: top-1 confidence {r['confidence']}, flagged={r['needs_manual_verification']}")

@software_test
def test_06_preprocessing_is_reproducible_and_consistent():
    need_samples()
    p = sample_paths[0]
    a1 = to_model_input(load_image_as_rgb_array(p)).numpy()
    a2 = to_model_input(load_image_as_rgb_array(p)).numpy()
    assert np.array_equal(a1, a2), "preprocessing is not deterministic"
    assert a1.shape == (224, 224, 3) and a1.dtype == np.uint8
    # training-style decode (TensorFlow) vs inference-style decode (Pillow) of the same file
    tf_img = tf.io.decode_image(open(p, "rb").read(), channels=3, expand_animations=False)
    b = to_model_input(tf_img).numpy()
    diff = np.abs(a1.astype(int) - b.astype(int))
    print(f"   Pillow vs TensorFlow decode: mean abs pixel diff = {diff.mean():.3f}, max = {diff.max()} (0-255 scale)")
    assert diff.mean() < 5
    # the in-model rescaling layer must equal the official MobileNetV2 preprocessing
    x = np.random.RandomState(1).randint(0, 256, (2, 224, 224, 3)).astype("float32")
    layer = rec_loaded.model.get_layer("preprocess_rescale")
    assert np.allclose(layer(x).numpy(), keras.applications.mobilenet_v2.preprocess_input(x.copy()), atol=1e-5)
    # grayscale and RGBA inputs must be converted to RGB and still work
    img = Image.open(p)
    for mode in ("L", "RGBA"):
        q = os.path.join(TMP, f"mode_{mode}.png"); img.convert(mode).save(q)
        r = rec_loaded.predict(q)
        assert r["status"] == "ok", (mode, r)

results_df = pd.DataFrame(TEST_LOG)
display(results_df)
print(results_df["result"].value_counts().to_dict())

## FastAPI integration preparation (design only, not built or tested here)

**Endpoint:** `POST /api/ai/food-recognition`   ·   **Content-Type:** `multipart/form-data` with one field `file` (the image)

**Server-side rules (to implement later)**
* Load `FoodRecognizer.from_dir(...)` **once at startup**, not per request.
* Reject files larger than 10 MB (read at most `MAX+1` bytes), check `content_type` ∈ {`image/jpeg`, `image/png`, `image/webp`, …}. Real validation is done by `decode_image_bytes` (it checks the actual image content, not just the filename).
* Use `recognizer.predict_bytes(data)`: no temp file path is ever involved, so none can leak.
* Return generic error messages (as `food_inference.py` already does). Log details server-side only.
* Run the model in a worker thread (`run_in_threadpool`) so it doesn't block the event loop.

**Success (HTTP 200)**: the values below are placeholders showing the *structure* only
```json
{
  "status": "ok",
  "predicted_label": "<label>",
  "predicted_food": "<label as text>",
  "food_category": "<category>",
  "category_needs_review": false,
  "confidence": 0.0,
  "top_predictions": [
    {"label": "<class_1>", "display_name": "<class 1>", "confidence": 0.0},
    {"label": "<class_2>", "display_name": "<class 2>", "confidence": 0.0},
    {"label": "<class_3>", "display_name": "<class 3>", "confidence": 0.0}
  ],
  "needs_manual_verification": true,
  "uncertainty_reasons": ["low_confidence"],
  "confidence_note": "Confidence is a raw softmax score, not a calibrated probability that the prediction is correct.",
  "disclaimer": "Food-type classification from a photo only. ...",
  "model_version": "<from model_config.json>"
}
```

**Errors** (body: `{"status": "error", "error_code": "...", "error_message": "..."}`)
| error_code | HTTP |
|---|---|
| `FILE_TOO_LARGE` | 413 |
| `UNSUPPORTED_FILE_TYPE` | 415 |
| `EMPTY_FILE`, `INVALID_IMAGE`, `IMAGE_TOO_SMALL` | 422 |
| `INFERENCE_ERROR`, `INTERNAL_ERROR` | 500 (generic message only) |

**Untested sketch of the route** (for orientation only):
```python
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from fastapi.concurrency import run_in_threadpool
from food_inference import FoodRecognizer, MAX_FILE_BYTES

app = FastAPI()
recognizer = FoodRecognizer.from_dir("food_model_artifacts")   # load once
STATUS = {"FILE_TOO_LARGE": 413, "UNSUPPORTED_FILE_TYPE": 415, "EMPTY_FILE": 422,
          "INVALID_IMAGE": 422, "IMAGE_TOO_SMALL": 422}

@app.post("/api/ai/food-recognition")
async def food_recognition(file: UploadFile = File(...)):
    data = await file.read(MAX_FILE_BYTES + 1)
    result = await run_in_threadpool(recognizer.predict_bytes, data)
    if result["status"] == "error":
        return JSONResponse(result, status_code=STATUS.get(result["error_code"], 500))
    return result
```

## Step 20 · Final project summary

Run the next cell for a summary generated from your **actual** variables and files, then read the checklist below it.

In [ ]:
print("=" * 72)
print("FINAL PROJECT SUMMARY" + ("   *** SMOKE TEST RUN: NUMBERS ARE NOT MEANINGFUL ***" if CFG["SMOKE_TEST"] else ""))
print("=" * 72)
print(f"Task            : Food-101 food-type classification ({NUM_CLASSES} classes), MobileNetV2 transfer learning")
print(f"Data splits     : train {int(train_mask.sum())} | validation {int(val_mask.sum())} | official test {len(test_labels_official)}"
      f" (evaluated on {EVAL['n_test_images']})")
print(f"Selected model  : {SELECTED_NAME} (chosen by VALIDATION loss)")
print("\nMeasured on the official test split (single evaluation):")
for k in ["test_loss", "test_accuracy", "top3_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]:
    print(f"  {k:18s} {EVAL[k]:.4f}")
u = EVAL["uncertainty_rule_on_test"]
print(f"\nUncertainty rule (conf < {u['conf_threshold']} or margin < {u['margin_threshold']}):")
print(f"  flags {u['fraction_flagged']:.1%} of test images for manual verification")
print(f"  accuracy when NOT flagged: {u['accuracy_when_not_flagged']:.3f} | when flagged: {u['accuracy_when_flagged']:.3f}")
print("\nSoftware tests:", pd.DataFrame(TEST_LOG)["result"].value_counts().to_dict())
print("\nArtifacts folder:", ARTIFACT_DIR)
print("\nREMINDERS (research integrity):")
print(" - These metrics come from Food-101 only; real donation photos will likely score lower.")
print(" - The model predicts food TYPE only: not quantity, freshness, remaining time or safety.")
print(" - Softmax confidence is not a calibrated probability; always allow manual correction.")
print(" - A Food-101 classifier alone is not a research contribution; your matching/redistribution")
print("   algorithm under time and demand uncertainty is a separate component.")

### Limitations to state in your report
* **Domain gap:** Food-101 is mostly single, plated, restaurant-style dishes; real surplus food (trays, bulk containers, packaged food, mixed dishes, poor lighting) differs. The reported accuracy is **not** an estimate of real-world accuracy. Evaluating on your own donor-style photos is future work.
* **Limited vocabulary:** only 101 classes with a Western/East-Asian bias. Common local dishes (e.g. many Indian dishes) are absent, so the model will map them to the nearest wrong class. Keep an "other / type it yourself" option in the app.
* **Noisy training labels** (by the dataset authors' design) can slightly depress validation numbers.
* **No quantity, freshness, remaining-time or safety information** is produced or implied.
* **Confidence ≠ probability.** Thresholds were tuned on validation data only and have no formal guarantee.
* **Not novel:** transfer learning on Food-101 is standard. It is a *component* of your system; the contribution has to come from the dynamic redistribution/matching approach.

---
## Implementation checklist

**Implemented in this notebook (code written and statically checked)**
- [x] 1 Environment/GPU check · 2 Imports, config, seeds · 3 Food-101 download via TFDS (source cited)
- [x] 4 Dataset inspection, class counts, sample image grid
- [x] 5 Stratified train/val split from the official train set, leakage assertions, alignment check; official test untouched until Step 12
- [x] 6 Preprocessing (RGB, 224×224, MobileNetV2 scaling) in one shared module; `tf.data` with batching, prefetch, validation caching, optional disk cache
- [x] 7 Training-only augmentation · 8 MobileNetV2 + GAP + Dropout + Dense(101, softmax), Adam, sparse CE, summary
- [x] 9 Head training: short check run, then conditional continuation, with EarlyStopping / ModelCheckpoint / ReduceLROnPlateau
- [x] 10 Fine-tuning (upper layers, BatchNorm frozen, smaller LR) · 11 Baseline vs fine-tuned compared on **validation**, best kept
- [x] 12–13 Test loss/accuracy, top-3, macro P/R/F1, weighted F1, per-class report, confusion matrix (full + labelled subset), curves
- [x] 14 Configurable category mapping with review flags and manual update function
- [x] 15 `predict_food()` with validation, top-3, uncertainty flag, error handling, manual correction, Colab upload cell
- [x] 16–17 Model/labels/mapping/config/metrics saved as JSON/Keras, optional Drive export · 18 Fresh-session reload cell
- [x] 19 Software tests (valid, invalid, low-confidence, structure, class count, preprocessing reproducibility) · FastAPI design

**Requires execution in Colab to verify (not yet run, so no results exist yet)**
- [ ] Food-101 download and split sizes (75,750 / 25,250 expected)
- [ ] Training behaviour, validation curves, whether fine-tuning beats the baseline
- [ ] **All accuracy / precision / recall / F1 / confusion-matrix numbers**
- [ ] The chosen uncertainty threshold and how it behaves on the test set
- [ ] All Step 19 software tests passing on your trained artifacts
- [ ] Review the category mapping yourself; it is a draft

## Step 21 · Add donation categories and package artifacts for the FastAPI service
The dish→donation grouping below is a **draft assumption**, editable in `ai_service/app/utils/donation_categories.json`. It is not a food-safety classification.

In [ ]:
DISH_TO_DONATION = {
    "dessert_sweet": "sweets_desserts",
    "breakfast_item": "bakery_dry",
    "egg_dish": "cooked_protein_meal",
    "meat_dish": "cooked_protein_meal",
    "seafood_dish": "cooked_protein_meal",
    "salad": "fresh_salad",
    "soup": "soup_liquid",
    "pasta_noodles": "cooked_meal",
    "rice_dish": "cooked_meal",
    "sandwich_burger_wrap": "ready_to_eat_snack",
    "pizza": "ready_to_eat_snack",
    "bread_bakery": "bakery_dry",
    "appetizer_snack_side": "ready_to_eat_snack",
    "prepared_dish": "cooked_meal"
}
for lab, entry in CATEGORY_MAP.items():
    entry["donation_category"] = DISH_TO_DONATION.get(entry["category"], "unmapped")
save_category_mapping()
print("Labels without donation_category:", [k for k, v in CATEGORY_MAP.items() if v["donation_category"] == "unmapped"])

# Re-export the mapping into the artifact folder and zip everything the service needs
needed = ["food_classifier_best.keras", "class_labels.json", "food_category_mapping.json", "model_config.json"]
missing = [f for f in needed if not os.path.exists(os.path.join(ARTIFACT_DIR, f))]
print("Missing artifacts (re-run Step 16 first):", missing)
if not missing:
    shutil.make_archive("/content/food_classifier_artifacts", "zip", ARTIFACT_DIR)
    print("Created /content/food_classifier_artifacts.zip -> unzip into ai_service/saved_models/food_classifier/")